### Das Model-View-Controller Pattern (MVC) am Beispiel eines Spiels
 
 1. **Das Model** (Game)
* Die Klasse Game verwaltet den Zustand des Spiels und die Spiellogik.
* Das Klasse Game ist komplett unabhängig von der Darstellung (View). Es weiß nicht, wie das Spiel gezeichnet wird (ob mit ipycanvas, in der Konsole oder 3D). Es signalisiert Änderungen durch Aufruf der
* registrierten  Callback-Funktionen.

2. **Die View**
* Die Klasse View kapselt das Canvas-Objekt und ist für die grafische Darstellung des Spielzustandes zuständig.
* Die View beobachtet das Klasse Game, indem es ein Callback beim Game registriert.
* Wenn sich der Spielzustand ändert, wird das von der View registrierte Callback
  aufgerufen und passt die Darstellung an. Dazu kann die View auch auf das Game-Objekt zugreifen.

3. **Der Controller**
* Der Controller benutzt das Canvas-Objekt der View um auf Tasten-und Mausevents zu hören.
* Der Controller nimmt die Eingaben des Benutzers entgegen und übersetzt diese in Befehle für das Model (die Methoden von Game).

<img src="/files/images/MCV.png">  

```python
from model_view_controller import Observable, notify, BaseView, Controller


class Game(Observable):
    ...

    @notify
    def action(...)
        ...
    


class View(BaseView):
    def __init__(self, game):
        super().__init__(game)

    def update(self, event, data):
        if event == 'some_event':
            self.handle_event(data)

            
game = Game()
callbacks = {...}
view = View(game)
controller = Controller(game, view, callbacks)
controller
```

### Beispiel
- Die Klasse `Game` speichert die Position des Spielers auf einem 4 x 5 Schachbrett.  
Die Methode `move(dx, dy)` bewegt, falls möglich, den Spieler um `dx` nach rechts und `dy` nach unten, und ruft die registrierten Callbacks auf.
- Die Klasse `View` erbt von `BaseView`.
  `super().__init__(...)` ruft die init-Methode von `BaseView` auf.
  Diese weist `self.mcanvas` und `self.canvas` eine MultiCanvas und deren oberster Layer zu.
  Weiter wird `update` bei der Game-Instanz als Callback registriert.
  Die View-Klasse benutzt die Klasse GridHelper, um das Gitter und den Spieler zu zeichnen.
- Der Controller nimmt u.a. einen Dict mit Callbacks als Argument.  
  Hier wird festgelegt, welche Funktionen beim Dücken der angegebenen Tasten des Mausbuttons aufgerufen werden.
  



In [ ]:
from model_view_controller import Observable, notify, BaseView, Controller
from gridhelper import GridHelper


class Game(Observable):
    def __init__(self):
        self.ncol = 4
        self.nrow = 5
        self.player_pos = (0, 0)

    def is_valid_pos(self, player_pos):
        col, row = player_pos
        return 0 <= col < self.ncol and 0 <= row < self.nrow

    @notify  # ruft registrierte Callbacks f mit f('move', <Return-Wert>) auf
    def move(self, dx, dy):
        new_player_pos = self.player_pos[0] + dx, self.player_pos[1] + dy
        if self.is_valid_pos(new_player_pos):
            self.player_pos = new_player_pos


class View(BaseView):
    def __init__(self, game, debug=True):
        super().__init__(game, debug=debug)  # self.mcanvas enthaelt MultiCanvas,
                                             # self.canvas ist oberster Layer von self.mcanvas
                                             # registriert self.update beim Game als Callback
        dx, dy = self.mcanvas.width/self.game.ncol, self.mcanvas.height/self.game.nrow
        ncol, nrow = self.game.ncol, self.game.nrow
        gridspec = (0, 0, dx, dy, ncol, nrow)
        self.gridhelper = GridHelper(*gridspec)

        self.redraw()

    def redraw(self):
        self.canvas.clear()
        self.gridhelper.draw_grid(self.canvas)
        self.gridhelper.fill_circle(self.canvas, self.game.player_pos)

    def update(self, event, data):
        self.redraw()


game = Game()


def on_mouse_down(self, x, y, state):
    col, row = self.game.player_pos
    col_new, row_new = self.view.gridhelper.xy2cr(x, y)
    dx, dy = col_new - col, row_new - row
    self.game.move(dx, dy)


callbacks = {'ArrowRight': lambda: game.move(1, 0),
             'ArrowLeft': lambda: game.move(-1, 0),
             'ArrowUp': lambda: game.move(0, -1),
             'ArrowDown': lambda: game.move(0, 1),
             'mouse_down': on_mouse_down,
             }

view = View(game, debug=False)
controller = Controller(game, view, callbacks, debug=False)
controller